In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [41]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [42]:
loader = PyPDFLoader("../data/1.Generative AI course.pdf")
docs = loader.load()
len(docs)

19

In [43]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap= 200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

31

In [44]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [45]:
vector_store = Chroma.from_documents(
    documents = splitted_data,
    embedding = embeddings
)

In [46]:
query = "Generative AI"
data = vector_store.similarity_search(query=query)

In [47]:
context = ""
for doc in data:
    context += doc.page_content + "\n"
    

In [71]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [72]:
#### Chain - context_generate | prompt | llm | strparser

In [74]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context": context,
        "question": query
    }

In [75]:
prompt = PromptTemplate.from_template("""
       You are a helpful assiatant  and provide answer based on the context for user question. and
       if you dont know the answer, then you can say that 'I dont know.'
       Context: {context}
       Question: {question}
    """)

In [76]:
rag_chain = get_context | prompt | llm

In [83]:
res = rag_chain.invoke("what is Java")

print(res.content)

I don't know.
